# EmotionLoRA — Train Any Adapter

**Runtime required:** T4 GPU (Runtime → Change runtime type → T4 GPU)

This notebook trains a LoRA adapter for any emotion on Mistral 7B and pushes it to HuggingFace Hub.

**To train a new emotion:** edit only Cell 1 (the config), then run all cells in order.

Steps:
1. **Config** — set emotion name, repo URL, HF token
2. Install dependencies
3. Clone the repo
4. Prepare training data
5. Train the adapter
6. Push to HuggingFace Hub
7. Build router embeddings
8. Verify

## Cell 1 — Config ✏️

**Edit this cell only.** Everything else runs automatically.

- `EMOTION` must match a file in `data/{emotion}_examples.jsonl` and an entry in `adapters/registry.json`
- `HF_TOKEN` needs write access — get one at huggingface.co/settings/tokens

In [ ]:
# ── Edit these three values, then run all cells ──────────────────────────────

EMOTION   = "empathy"       # e.g. empathy, encouragement, playfulness, warmth, frustration, firmness
REPO_URL  = "https://github.com/ismailahmedsh/LoraEmotion.git"
HF_TOKEN  = "hf_your_token_here"  # HuggingFace write token

# ─────────────────────────────────────────────────────────────────────────────
print(f"Config: emotion='{EMOTION}', repo='{REPO_URL}'")

## Cell 2 — Install dependencies

Unsloth must be installed before any other imports. Takes ~2 minutes on first run.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl transformers peft datasets sentence-transformers python-dotenv -q
print('Dependencies installed.')

## Cell 3 — Clone the repo

Pulls the latest code and data into `/content/emotionlora`.

In [ ]:
import os

REPO_DIR = "/content/emotionlora"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

# Write HF token to .env so training scripts can read it
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

print(f"Working directory: {os.getcwd()}")
print(f"Training data file: data/{EMOTION}_examples.jsonl")

if not os.path.exists(f"data/{EMOTION}_examples.jsonl"):
    raise FileNotFoundError(f"data/{EMOTION}_examples.jsonl not found. Check the EMOTION name in Cell 1.")

import json
registry = json.load(open("adapters/registry.json"))
if EMOTION not in registry:
    raise KeyError(f"'{EMOTION}' not found in adapters/registry.json. Add the entry before training.")

print(f"Registry entry: {registry[EMOTION]}")
print("\nReady to train.")

## Cell 4 — Prepare training data

Converts `data/{emotion}_examples.jsonl` → HuggingFace Dataset with Alpaca format.

In [ ]:
!python data/prepare.py --emotion {EMOTION}

## Cell 5 — Train the adapter

Loads Mistral 7B (4-bit), attaches LoRA, trains for 60 steps (~5–10 minutes on T4).

Watch the loss — it should decrease over steps. Final loss around 0.5–1.5 is healthy for this dataset size.

In [ ]:
!python training/train_adapter.py --emotion {EMOTION}

## Cell 6 — Push adapter to HuggingFace Hub

Uploads only the LoRA adapter weights — not the full base model (which stays on HF).

In [ ]:
!python training/push_to_hub.py --emotion {EMOTION}

## Cell 7 — Build router embeddings

Generates the centroid embedding for this emotion so the router can match messages to it.
Saves to `router/embeddings/{emotion}.npy`.

This step is required for the router to recognise this emotion. Run it every time you add or retrain an adapter.

In [ ]:
!python router/embeddings/build_embeddings.py --emotion {EMOTION}

## Cell 8 — Verify

Confirms the adapter is live on the Hub, the registry entry is correct, and the embedding was built.

In [ ]:
import json
import os
from huggingface_hub import list_repo_files

registry = json.load(open("adapters/registry.json"))
entry = registry[EMOTION]
repo_id = entry["repo_id"]

print(f"=== Verifying: {EMOTION} ===")
print(f"Registry entry: {json.dumps(entry, indent=2)}")

# Check HF Hub
print(f"\nChecking HuggingFace Hub: {repo_id}")
try:
    files = list(list_repo_files(repo_id))
    if files:
        print("Files on Hub:")
        for f in files:
            print(f"  {f}")
        print(f"\nAdapter live at: https://huggingface.co/{repo_id}")
    else:
        print("WARNING: No files found on Hub — push may have failed.")
except Exception as e:
    print(f"ERROR checking Hub: {e}")

# Check embedding
npy_path = f"router/embeddings/{EMOTION}.npy"
if os.path.exists(npy_path):
    import numpy as np
    vec = np.load(npy_path)
    print(f"\nEmbedding: {npy_path} — shape {vec.shape}, L2 norm {np.linalg.norm(vec):.4f}")
else:
    print(f"\nWARNING: {npy_path} not found — Cell 7 may have failed.")

print("\n=== Done ===")